# Experimental iodine-bearing Figshare augmentation

Use the previous nine-element Zn experiment as the baseline. Fit a fresh Zn ExpertXAS with widths **310–500–750**, a UniversalXAS with extra iodine-bearing XANES, and the usual Zn-tuned head initialized from the selected UniversalXAS.

Run in the OmniXAS Python 3.11 training environment with the full Figshare XANES extraction and the saved nine-element dataset. Set `RUN_TRAINING = True` to retrain and evaluate. Training can take substantial GPU time. This experiment keeps its data, best weights, and results in the notebook kernel. Plot output stays inline.

The original nine-element material assignments remain fixed. The already-inspected Zn test set makes this comparison exploratory.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
from unittest.mock import patch
import csv
import hashlib
import json
import os
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "tutorial_omnixas"))
os.environ.setdefault("DGLBACKEND", "pytorch")
sys.dont_write_bytecode = True

DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
RAW_ROOT = DATA_ROOT / "figshare"
ELEMENTS = ("Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn")
TASKS = [f"{element}_FEFF" for element in ELEMENTS]
SPLITS = ("train", "val", "test")
SPLIT_SEED = 42
ENCODER_SEED = 123
RUN_TRAINING = False

EXPERT_WIDTHS = (310, 500, 750)
UNIVERSAL_WIDTHS = (500, 500, 550)
EXPERT_HP = {"lr": 0.002, "dropout": 0.05, "schedule": "plateau", "seed": 43}
UNIVERSAL_TRIALS, TUNED_TRIALS = 100, 300
ENCODER_EPOCHS, EXPERT_EPOCHS = 1000, 800
UNIVERSAL_EPOCHS, TUNED_EPOCHS = 800, 1000
BATCH_SIZE, GRAPH_BATCH_SIZE, PATIENCE = 4096, 32, 60
ENERGY_GRID = np.linspace(9659.0, 9694.0, 141)
RELATIVE_GRID = np.linspace(-10.0, 25.0, 141)
SIGMA_EV = 0.5

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def split_id(value):
    material, site = value.rsplit("_", 1)
    return material, int(site)

## Build the iodine cohort

Scan the full Figshare extraction across all absorbers. Use singly matched K-edge XANES with iodine in the archived structure. Include eligible site keys outside the original whitelist. Exact site keys already in the nine-element dataset count once.

Keep each material's existing split if any of its sites already appears in the nine-element dataset. Assign new materials by seeded 80/10/10, with all absorbers and sites on a material assigned together.

For added Ti–Cu spectra, use the existing unscaled exporter. For added Zn spectra, use the existing Zn absolute grid. Other absorbers use unscaled column-four intensity on a 141-point grid from −10 to +25 eV using the source's relative energy in column two. [FEFF output columns](https://feff.phys.washington.edu/feff/wiki/static/f/e/f/FEFF_Output_files_583c.html)

In [ ]:
from export_figshare_141_splits_unscaled import ELEMENT_START_EV, ExportError, read_spectral_key_file, sample_spectrum

dataset_file = DATASET_ROOT / "dataset.json"
dataset_hash = sha256(dataset_file)
dataset_info = json.loads(dataset_file.read_text(encoding="utf-8"))
if dataset_info.get("target_scale") != 1 or dataset_info.get("feature_scale") != 1000:
    raise ValueError("Expected the previous unscaled nine-element dataset and 1000× feature scale.")
if not np.array_equal(dataset_info.get("zn_grid_ev"), ENERGY_GRID):
    raise ValueError("The saved Zn grid differs from the previous experiment.")

base = {}
key_split, material_split = {}, {}
for element in ELEMENTS:
    for split in SPLITS:
        prefix = f"{element}_FEFF_{split}"
        ids = (DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt").read_text(encoding="utf-8-sig").splitlines()
        y = np.loadtxt(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
        if not ids or y.shape != (len(ids), 141) or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError(f"Invalid canonical IDs or targets for {prefix}.")
        keys = [(element, *split_id(value)) for value in ids]
        for key in keys:
            _, material, site = key
            if not material or site < 0 or key in key_split:
                raise ValueError(f"Invalid or duplicate canonical key {key}.")
            key_split[key] = split
            if material_split.setdefault(material, split) != split:
                raise ValueError(f"Material appears in multiple canonical splits: {material}")
        base[element, split] = (ids, keys, y)

raw_manifest = RAW_ROOT / "manifest.csv"
raw_report = json.loads((RAW_ROOT / "report.json").read_text(encoding="utf-8"))
if raw_report.get("figshare_archive_file_id") != 9932248 or not raw_report.get("spectrum_type_used", "").startswith("XANES only"):
    raise ValueError(f"Expected the full XANES extraction from extract_figshare_dataset.py at {RAW_ROOT}.")
whitelist_path = REPO_ROOT / "tutorial_omnixas/final_spectral_npz_keys.tsv"
whitelist = read_spectral_key_file(whitelist_path)

def has_iodine(directory):
    poscar = directory / "POSCAR"
    if poscar.is_file():
        with poscar.open("rb") as handle:
            header = [handle.readline().split() for _ in range(7)]
        symbols, counts = header[5:]
        if not symbols or len(symbols) != len(counts):
            raise ValueError(f"Invalid POSCAR header: {poscar}")
        return any(symbol == b"I" and int(count) > 0 for symbol, count in zip(symbols, counts, strict=True))
    structure = json.loads((directory / "structure.json").read_bytes())
    return any(atom["element"] == "I" and atom.get("occu", 1) > 0
               for site in structure["sites"] for atom in site["species"])

canonical_grids = {element: np.linspace(start, start + 35.0, 141) for element, start in ELEMENT_START_EV.items()}

def resample_added(path, absorber):
    if absorber in canonical_grids:
        return sample_spectrum(path, absorber).astype(np.float32), canonical_grids[absorber]
    grid = ENERGY_GRID if absorber == "Zn" else RELATIVE_GRID
    table = np.loadtxt(path, ndmin=2)
    if table.shape[1] < 4 or not len(table):
        raise ValueError("Expected a nonempty source table with at least four columns.")
    if absorber != "Zn":
        # The two FEFF energy columns should differ by the same reference energy.
        offset = table[:, 0] - table[:, 1]
        if not np.isfinite(offset).all() or np.ptp(offset) > 0.05:
            raise ValueError("Inconsistent absolute and relative energy columns.")
    energy, mu = table[:, 0 if absorber == "Zn" else 1], table[:, 3]
    keep = np.isfinite(energy) & np.isfinite(mu) & (mu > 0)
    if not keep.any():
        raise ValueError("No finite positive source samples.")
    order = np.argsort(energy[keep], kind="stable")
    energy, mu = energy[keep][order], mu[keep][order]
    energy, starts, counts = np.unique(energy, return_index=True, return_counts=True)
    mu = np.add.reduceat(mu, starts) / counts
    inside = (energy > grid[0]) & (energy < grid[-1])
    if absorber == "Zn" and not inside.any():
        return np.full(141, mu[np.argmin(abs(energy - grid[0]))], dtype=np.float32), grid
    if not inside.any():
        raise ValueError("No positive source samples in the relative-energy window.")
    return np.interp(grid, energy[inside], mu[inside]).astype(np.float32), grid

In [ ]:
base_iodine = set()
candidates = []
omissions = []
seen = set()
with raw_manifest.open(encoding="utf-8-sig", newline="") as handle:
    for record in csv.DictReader(handle):
        absorber, material, site = record["element"], record["material_id"], int(record["site"])
        key = absorber, material, site
        if key in seen:
            raise ValueError(f"Duplicate key in raw manifest: {key}")
        seen.add(key)
        directory = RAW_ROOT / "FEFF" / absorber / material / "FEFF-XANES" / f"{site:03d}_{absorber}"
        if record["site_directory"] != directory.relative_to(RAW_ROOT).as_posix():
            raise ValueError(f"Unexpected Figshare directory for {key}.")
        if record["status"] != "matched" or int(record["raw_candidate_count"]) != 1:
            if key in key_split:
                raise ValueError(f"Canonical site lacks a unique raw source: {key}")
            if record["status"] == "ambiguous":
                candidate_dirs = (directory / "candidates").iterdir()
                if any(has_iodine(path) for path in candidate_dirs if path.is_dir()):
                    omissions.append((*key, "ambiguous XANES"))
            continue
        if not has_iodine(directory):
            continue
        if key in key_split:
            base_iodine.add(key)
            continue
        source = json.loads((directory / "figshare_record.json").read_bytes())
        if str(source.get("edge", "")).upper() != "K" or str(source.get("spectrum_type", "")).upper() != "XANES":
            omissions.append((*key, "outside K-edge XANES"))
            continue
        try:
            y, grid = resample_added(directory / "spectrum_raw.dat", absorber)
        except (ValueError, ExportError) as exc:
            omissions.append((*key, str(exc)))
            continue
        if y.shape != (141,) or not np.isfinite(y).all() or np.any(y < 0):
            raise ValueError(f"Invalid experimental target for {key}")
        candidates.append((key, directory / "structure.json", y, grid))

if len(seen) != raw_report["row_count"] or key_split.keys() - seen:
    raise ValueError("The raw manifest did not cover its extraction report and all canonical IDs.")
if not base_iodine or not candidates:
    raise ValueError("Expected iodine-bearing rows in both the canonical dataset and the full extraction.")
print(f"{len(seen):,} raw sites scanned, {len(base_iodine):,} iodine-bearing canonical sites, "
      f"{len(candidates):,} additional candidate sites.")
display(pd.DataFrame(Counter(row[-1] for row in omissions).most_common(), columns=["Reason", "Sites"]))

## Retrain the standard nine-element encoder

Use the scratch M3GNet configuration from the original Zn sweep, including nine-element balanced batches, the spectral-difference loss term, seed 123, and validation macro-relative median MSE. The iodine cohort enters only after the best encoder has been selected and frozen. The encoder checkpoint and exported features stay in memory.

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError("The input audit is complete. Set RUN_TRAINING = True and rerun to train.")

from io import BytesIO
from tempfile import TemporaryDirectory

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pymatgen.core import Structure
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import omnixas.data.feff_graph as graph_data
from omnixas.data.feff_graph import (
    BalancedTaskBatchSampler, CollateGraphs, FEFFDataset,
    patch_matgl_gpu_constants, validate_raw_structures,
)
from omnixas.model.m3gnet_xas import M3GNetXAS
from omnixas.model.xasblock import XASBlock
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEFF_TASKS = TASKS
ENCODER_MONITOR = "val_balanced_rel_mse"

def snapshot(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def state_sha256(state):
    buffer = BytesIO()
    torch.save(state, buffer)
    return hashlib.sha256(buffer.getbuffer()).hexdigest()

def encode_chunk(model, collator, rows):
    batch = collator(rows)
    return encode_batch(model, batch, len(rows))

def encode_batch(model, batch, expected_rows):
    with torch.inference_mode():
        X = model.encode(batch["graph"].to(device), batch["line_graph"].to(device),
                         batch["site"].to(device), scaled=True).cpu().numpy()
    if X.shape != (expected_rows, 64) or not np.isfinite(X).all():
        raise ValueError("Invalid encoded feature batch.")
    return X

class LitScratch(pl.LightningModule):
    def __init__(self, model: M3GNetXAS, train_base: torch.Tensor, val_base: torch.Tensor, lr: float):
        super().__init__()
        self.model, self.lr = model, lr
        self.register_buffer("train_base", train_base)
        self.register_buffer("val_base", val_base)
        self.val_mse, self.val_task = [], []

    def step(self, batch, stage):
        graph = batch["graph"].to(self.device)
        line_graph = batch["line_graph"].to(self.device)
        site = batch["site"].to(self.device)
        pred = self.model(graph, line_graph, site)
        y = batch["y"].to(self.device)
        task = batch["task"].to(self.device)
        mse = ((pred - y) ** 2).mean(1)
        base = self.train_base if stage == "train" else self.val_base
        loss = (mse / base[task].clamp_min(1e-12)).mean() + 0.02 * (
            torch.diff(pred, dim=1) - torch.diff(y, dim=1)
        ).square().mean()
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, batch_size=y.shape[0])
        if stage == "val":
            self.val_mse.append(mse.detach())
            self.val_task.append(task.detach())
        return loss

    def training_step(self, batch, _):
        return self.step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mse, self.val_task = [], []

    def validation_step(self, batch, _):
        return self.step(batch, "val")

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking or not self.val_mse:
            return
        mses, tasks = torch.cat(self.val_mse), torch.cat(self.val_task)
        rel = [
            mses[tasks == i].median() / self.val_base[i].clamp_min(1e-12)
            for i in range(len(FEFF_TASKS))
            if (tasks == i).any()
        ]
        if len(rel) != len(FEFF_TASKS):
            raise RuntimeError("Validation lacks one or more FEFF elements")
        self.log("val_balanced_rel_mse", torch.stack(rel).mean(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-5,
            fused=self.device.type == "cuda",
        )
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=16, min_lr=1e-6)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val_balanced_rel_mse"},
        }

class BestEncoderInMemory(pl.Callback):
    def __init__(self):
        self.metric, self.epoch, self.state = float("inf"), None, None

    def on_validation_end(self, trainer, module):
        if trainer.sanity_checking:
            return
        value = float(trainer.callback_metrics[ENCODER_MONITOR])
        if not np.isfinite(value):
            raise ValueError("Non-finite encoder validation metric.")
        if value < self.metric:
            self.metric, self.epoch = value, trainer.current_epoch
            self.state = snapshot(module.model)

def fit_encoder():
    patch_matgl_gpu_constants()
    validate_raw_structures(REPO_ROOT, RAW_ROOT, TASKS, DATASET_ROOT)
    pl.seed_everything(ENCODER_SEED, workers=True)
    torch.set_float32_matmul_precision("high")
    model = M3GNetXAS(head_dropout=0.05)
    bases = [torch.tensor([np.median(
        ((base[element, split][2] - base[element, "train"][2].mean(0)) ** 2).mean(1)
    ) for element in ELEMENTS]) for split in ("train", "val")]
    best = BestEncoderInMemory()

    with patch.object(graph_data, "FEFF_TASKS", TASKS), TemporaryDirectory(prefix="omnixas-iodine-") as scratch:
        collator = CollateGraphs(model.encoder)
        train = FEFFDataset(REPO_ROOT, RAW_ROOT, TASKS, "train", DATASET_ROOT)
        val = FEFFDataset(REPO_ROOT, RAW_ROOT, TASKS, "val", DATASET_ROOT)
        train_counts = Counter(row[0] for row in train.rows)
        sampler = BalancedTaskBatchSampler(train.rows, min(64, min(train_counts.values())), ENCODER_SEED)
        train_loader = DataLoader(train, batch_sampler=sampler, collate_fn=collator, num_workers=0)
        val_loader = DataLoader(val, batch_size=1024, collate_fn=collator, num_workers=0)
        trainer = pl.Trainer(
            max_epochs=ENCODER_EPOCHS, accelerator="auto", devices=1,
            precision="bf16-mixed" if device.type == "cuda" and torch.cuda.is_bf16_supported() else "32-true",
            callbacks=[best, EarlyStopping(monitor=ENCODER_MONITOR, patience=60, mode="min")],
            logger=False, enable_checkpointing=False, default_root_dir=scratch,
        )
        trainer.fit(LitScratch(model, *bases, 0.001), train_loader, val_loader)

    if best.state is None:
        raise RuntimeError("No encoder validation checkpoint was selected.")
    model.load_state_dict(best.state, strict=True)
    model.to(device).eval().requires_grad_(False)
    return model, state_sha256(best.state), {"seed": ENCODER_SEED, "epoch": best.epoch, "validation": best.metric}

def export_base_features(model):
    data = {}
    with patch.object(graph_data, "FEFF_TASKS", TASKS):
        collator = CollateGraphs(model.encoder)
    for (element, split), (_, keys, reference_y) in tqdm(base.items(), desc="Encode the nine-element splits"):
        dataset = FEFFDataset(REPO_ROOT, RAW_ROOT, [f"{element}_FEFF"], split, DATASET_ROOT)
        source_keys = [(element, material, site) for _, material, site, _ in dataset.rows]
        if source_keys != keys:
            raise ValueError(f"Feature rows are out of order: {element} {split}")
        loader = DataLoader(dataset, batch_size=1024, collate_fn=collator, num_workers=0)
        xs, ys = [], []
        for batch in loader:
            xs.append(encode_batch(model, batch, len(batch["y"])))
            ys.append(batch["y"].numpy())
        X, y = np.concatenate(xs), np.concatenate(ys)
        if X.shape != (len(keys), 64) or not np.array_equal(y, reference_y):
            raise ValueError(f"Encoded rows do not align with canonical targets: {element} {split}")
        data[element, split] = X, y
        collator._graph_cache.clear()
    return data

encoder_model, encoder_hash, encoder_summary = fit_encoder()
base_features = export_base_features(encoder_model)
display(pd.DataFrame([encoder_summary]))

In [ ]:
vocabulary = set(encoder_model.encoder.element_types)
extra_tasks = sorted({f"{key[0]}_FEFF" for key, _, _, _ in candidates})
with patch.object(graph_data, "FEFF_TASKS", extra_tasks):
    collator = CollateGraphs(encoder_model.encoder)

def encode_added(encoder, collator, candidates, vocabulary):
    good, removed = [], []
    for start in tqdm(range(0, len(candidates), GRAPH_BATCH_SIZE), desc="Encode added iodine sites"):
        rows, metadata = [], []
        for key, path, y, grid in candidates[start:start + GRAPH_BATCH_SIZE]:
            absorber, material, site = key
            structure = Structure.from_dict(json.loads(path.read_bytes()))
            if not structure.is_ordered:
                removed.append((*key, "disordered structure"))
                continue
            elements = {element.symbol for element in structure.composition.elements}
            if "I" not in elements or site >= len(structure) or site < 0 or structure[site].specie.symbol != absorber:
                raise ValueError(f"Archived species or site disagrees with the raw key: {key}")
            if elements - vocabulary:
                removed.append((*key, "outside encoder element vocabulary"))
                continue
            rows.append((f"{absorber}_FEFF", material, structure, site, torch.from_numpy(y)))
            metadata.append((key, y, grid))
        if rows:
            X = encode_chunk(encoder, collator, rows)
            good.extend((key, x, y, grid) for (key, y, grid), x in zip(metadata, X, strict=True))
        collator._graph_cache.clear()
    return good, removed

extra_rows, structure_omissions = encode_added(encoder_model, collator, candidates, vocabulary)
omissions = omissions + structure_omissions

def assign_extra_splits(rows, existing, seed):
    combined = existing.copy()
    new = sorted({key[1] for key, *_ in rows} - existing.keys())
    if len(new) < 3:
        raise ValueError("At least three new materials are required for three experimental splits.")
    rng = np.random.default_rng(seed)
    rng.shuffle(new)
    n_train = min(len(new) - 2, max(1, round(0.8 * len(new))))
    n_val = min(len(new) - n_train - 1, max(1, round(0.1 * len(new))))
    for split, materials in zip(SPLITS, (new[:n_train], new[n_train:n_train + n_val], new[n_train + n_val:]), strict=True):
        combined.update(dict.fromkeys(materials, split))
    return combined

assignments = assign_extra_splits(extra_rows, material_split, SPLIT_SEED)
extra = {split: [] for split in SPLITS}
for key, X, y, grid in extra_rows:
    extra[assignments[key[1]]].append((key, X, y, grid))
if not all(extra.values()):
    raise ValueError("The encoded iodine cohort does not cover all three splits.")

audit = []
for split in SPLITS:
    for key, *_ in extra[split]:
        element, material, site = key
        audit.append({"absorber": element, "material": material, "site": site, "split": split,
                      "on original whitelist": key in whitelist, "canonical material": material in material_split})
iodine_audit = pd.DataFrame(audit)
display(iodine_audit.groupby("split").agg(sites=("site", "size"), materials=("material", "nunique")).reindex(SPLITS))
display(pd.crosstab(iodine_audit["absorber"], iodine_audit["split"]).reindex(columns=SPLITS, fill_value=0))
display(pd.DataFrame(Counter(row[-1] for row in omissions).most_common(), columns=["Omission reason", "Sites"]))
print(f"Fresh encoder state SHA-256: {encoder_hash}\n"
      f"Dataset SHA-256: {dataset_hash}\nRaw manifest SHA-256: {sha256(raw_manifest)}\n"
      f"Split seed: {SPLIT_SEED}. Feature scale: 1000. Target scale: 1.")

## Train ExpertXAS, augmented UniversalXAS, and Tuned-UniversalXAS

Fit ExpertXAS and Tuned-UniversalXAS on canonical Zn train rows. Keep the nine task balances from the latest notebook for UniversalXAS, then add the iodine train rows once as a tenth loss group. Use the previous loss scaling and search settings. All three heads select by **canonical Zn validation median MSE**. The new iodine validation and test rows provide diagnostics after selection.

The best state for each stage stays in memory. Tuning can keep the original UniversalXAS weights if no Zn fine-tuning epoch improves validation.

In [ ]:
def make_head(widths, dropout, state=None):
    with patch.object(XASBlock, "DROPOUT", dropout):
        model = XASBlock(64, list(widths), 141).to(device)
    if state is not None:
        model.load_state_dict(state, strict=True)
    return model

def predict_head(model, X):
    model.eval()
    with torch.inference_mode():
        predictions = torch.cat([model(batch.to(device)).cpu()
                                 for batch in torch.from_numpy(X).split(BATCH_SIZE)]).numpy()
    if predictions.shape != (len(X), 141) or not np.isfinite(predictions).all():
        raise ValueError("Invalid prediction batch.")
    return predictions

def training_baseline(y):
    return float(np.median(((y - y.mean(axis=0)) ** 2).mean(axis=1)))

rng = np.random.default_rng(SPLIT_SEED)
n_per_task = max(len(base_features[element, "train"][0]) for element in ELEMENTS)
balanced_X, balanced_y, balanced_tasks, loss_baselines = [], [], [], []
for task_id, element in enumerate(ELEMENTS):
    X, y = base_features[element, "train"]
    indices = np.r_[np.arange(len(X)), rng.integers(len(X), size=n_per_task - len(X))]
    rng.shuffle(indices)
    balanced_X.append(X[indices])
    balanced_y.append(y[indices])
    balanced_tasks.append(np.full(len(indices), task_id, dtype=np.int64))
    loss_baselines.append(training_baseline(y))

iodine_X = np.stack([row[1] for row in extra["train"]])
iodine_y = np.stack([row[2] for row in extra["train"]])
balanced_X.append(iodine_X)
balanced_y.append(iodine_y)
balanced_tasks.append(np.full(len(iodine_X), len(ELEMENTS), dtype=np.int64))
loss_baselines.append(training_baseline(iodine_y))

universal_X = torch.as_tensor(np.concatenate(balanced_X), device=device)
universal_y = torch.as_tensor(np.concatenate(balanced_y), device=device)
universal_groups = torch.as_tensor(np.concatenate(balanced_tasks), device=device)
group_baselines = torch.as_tensor(loss_baselines, dtype=torch.float32, device=device).clamp_min(1e-12)
zn_train_X, zn_train_y = (torch.as_tensor(arr, device=device) for arr in base_features["Zn", "train"])
zn_val_X, zn_val_y = (torch.as_tensor(arr, device=device) for arr in base_features["Zn", "val"])

def fit_head(X, y, widths, hp, epochs, *, source=None, groups=None):
    random.seed(hp["seed"])
    np.random.seed(hp["seed"])
    torch.manual_seed(hp["seed"])
    if device.type == "cuda":
        torch.cuda.manual_seed_all(hp["seed"])
    if len(X) < 2:
        raise ValueError("BatchNorm requires at least two training examples.")
    model = make_head(widths, hp["dropout"], source)
    optimizer = torch.optim.Adam(model.parameters(), lr=hp["lr"], fused=device.type == "cuda")
    plateau = hp["schedule"] == "plateau"
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=16, min_lr=1e-6)
                 if plateau else torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6))

    def validation_median():
        model.eval()
        with torch.inference_mode():
            median = float(torch.quantile((model(zn_val_X) - zn_val_y).square().mean(1), 0.5))
        if not np.isfinite(median):
            raise ValueError("Non-finite Zn validation median MSE.")
        return median

    best, best_epoch, best_state = float("inf"), None, None
    if source is not None:
        best, best_state = validation_median(), snapshot(model)
        if plateau:
            scheduler.step(best)
    stale = 0
    for epoch in range(1, epochs + 1):
        model.train()
        batches = list(torch.randperm(len(X), device=device).split(BATCH_SIZE))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2:] = [torch.cat(batches[-2:])]
        for index in batches:
            optimizer.zero_grad(set_to_none=True)
            errors = (model(X[index]) - y[index]).square().mean(1)
            if groups is None:
                loss = errors.mean()
            else:
                group = groups[index]
                loss = torch.stack([errors[group == i].mean() / group_baselines[i].pow(1.5)
                                    for i in torch.unique(group)]).mean()
            loss.backward()
            optimizer.step()
        metric = validation_median()
        if metric < best:
            best, best_epoch, best_state = metric, epoch, snapshot(model)
            stale = 0
        else:
            stale += 1
        if plateau:
            scheduler.step(metric)
        else:
            scheduler.step()
        if stale >= PATIENCE:
            break
    if best_state is None:
        raise RuntimeError("No head validation checkpoint was selected.")
    return best_state, {**hp, "Zn validation median MSE": best, "best epoch": best_epoch, "epochs run": epoch}

def trial_settings(count, seed, max_lr):
    rng = np.random.default_rng(seed)
    return [{"lr": float(np.exp(rng.uniform(np.log(1e-5), np.log(max_lr)))),
             "dropout": float(rng.choice([0.0, 0.05, 0.10, 0.20])),
             "schedule": str(rng.choice(["plateau", "cosine"])),
             "seed": int(rng.integers(1_000_000))} for _ in range(count)]

def sweep(label, X, y, settings, epochs, *, source=None, groups=None):
    winner, results = None, []
    progress = tqdm(settings, desc=label)
    for trial, hp in enumerate(progress, 1):
        state, record = fit_head(X, y, UNIVERSAL_WIDTHS, hp, epochs, source=source, groups=groups)
        record = {"trial": trial, **record}
        results.append(record)
        if winner is None or record["Zn validation median MSE"] < winner[1]["Zn validation median MSE"]:
            winner = state, record
        progress.set_postfix(best=winner[1]["Zn validation median MSE"])
    return winner, pd.DataFrame(results)

In [ ]:
expert_state, expert_config = fit_head(zn_train_X, zn_train_y, EXPERT_WIDTHS, EXPERT_HP, EXPERT_EPOCHS)
(universal_state, universal_config), universal_trials = sweep(
    "UniversalXAS", universal_X, universal_y, trial_settings(UNIVERSAL_TRIALS, 1042, 1e-2),
    UNIVERSAL_EPOCHS, groups=universal_groups,
)
(tuned_state, tuned_config), tuned_trials = sweep(
    "Tuned-UniversalXAS", zn_train_X, zn_train_y, trial_settings(TUNED_TRIALS, 2042, 5e-3),
    TUNED_EPOCHS, source=universal_state,
)

model_heads = {
    "ExpertXAS": make_head(EXPERT_WIDTHS, expert_config["dropout"], expert_state),
    "UniversalXAS": make_head(UNIVERSAL_WIDTHS, universal_config["dropout"], universal_state),
    "Tuned-UniversalXAS": make_head(UNIVERSAL_WIDTHS, tuned_config["dropout"], tuned_state),
}
display(pd.DataFrame([expert_config, universal_config, tuned_config], index=model_heads))
print(f"UniversalXAS initialization for tuning: {state_sha256(universal_state)}")
if tuned_config["best epoch"] is None:
    print("The selected tuned head retains the UniversalXAS weights.")

## Evaluate on the original Zn test set

Compare the three heads on identical Zn test spectra. Compute eta from the canonical Zn training-mean baseline. Show the error-versus-detail plots, the held-out iodine outlier, the two percentile grids, and the iodine-containing Zn comparison.

In [ ]:
from scipy.stats import pearsonr, spearmanr

test_ids, test_keys, test_y = base["Zn", "test"]
test_X, _ = base_features["Zn", "test"]
train_mean = base["Zn", "train"][2].mean(axis=0)
model_predictions = {name: predict_head(model, test_X) for name, model in model_heads.items()}
model_errors = {name: ((prediction - test_y) ** 2).mean(axis=1)
                for name, prediction in model_predictions.items()}
errors = model_errors["ExpertXAS"]
baseline_errors = ((test_y - train_mean) ** 2).mean(axis=1)

def error_ratio(baseline, model):
    return float(baseline / model) if model > 0 else (float("inf") if baseline > 0 else float("nan"))

comparison = pd.DataFrame([
    {"model": name, "head parameters": sum(p.numel() for p in model_heads[name].parameters()),
     "test MSE": values.mean(), "test median MSE": np.median(values),
     "test eta": error_ratio(np.median(baseline_errors), np.median(values))}
    for name, values in model_errors.items()
])
display(comparison)

def noise_ratio(y):
    scale = float(np.std(y))
    if scale == 0:
        return 0.0
    sigma = SIGMA_EV / (ENERGY_GRID[1] - ENERGY_GRID[0])
    half = max(1, int(3 * sigma))
    x = np.arange(-half, half + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= kernel.sum()
    smooth = np.convolve(np.pad(y, half, mode="edge"), kernel, mode="valid")
    return float(np.std(y - smooth) / scale)

ratios = np.array([noise_ratio(y) for y in test_y])
outlier_key = ("Zn", "mp-571292", 0)
if outlier_key not in test_keys:
    raise ValueError("The canonical Zn iodine outlier mp-571292 site 000 is missing.")
outlier = test_keys.index(outlier_key)
epsilon = np.finfo(float).eps
pooled = np.concatenate(list(model_errors.values()))
xmax, ymax = max(float(np.percentile(pooled, 99)), epsilon), max(float(np.percentile(ratios, 99)), epsilon)
names = list(model_errors)

fig, axes = plt.subplots(2, len(names), figsize=(5.1 * len(names), 8), sharey="row", squeeze=False)
for column, name in enumerate(names):
    values = model_errors[name]
    for row, ax in enumerate(axes[:, column]):
        ax.scatter(values, ratios, s=14, alpha=0.5)
        ax.scatter(values[outlier], ratios[outlier], s=55, c="tab:red", zorder=3)
        ax.set_xlabel("Per-spectrum test MSE")
        ax.grid(alpha=0.2)
        if row == 0:
            ax.set_xlim(0, max(float(pooled.max()) * 1.05, epsilon))
        else:
            ax.set_xlim(0, xmax)
            ax.set_ylim(0, ymax)
    if np.ptp(values) and np.ptp(ratios):
        pearson, spearman = pearsonr(values, ratios).statistic, spearmanr(values, ratios).statistic
        axes[0, column].set_title(f"{name} | Pearson {pearson:.2f} | Spearman {spearman:.2f}")
    else:
        axes[0, column].set_title(name)
axes[0, 0].set_ylabel("Detail ratio, all test rows")
axes[1, 0].set_ylabel("Detail ratio, pooled 99th percentile zoom")
fig.suptitle("Zn test error and spectral detail | red marks mp-571292 site 000")
fig.tight_layout()
display(fig)
plt.close(fig)

def plot_spectra(ax, grid, y, predictions, title, xlabel="Energy (eV)"):
    ax.plot(grid, y, color="black", linewidth=1.5, label="Target")
    for name, prediction in predictions.items():
        ax.plot(grid, prediction, label=name)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Intensity (unscaled)")
    ax.grid(alpha=0.2)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
plot_spectra(ax, ENERGY_GRID, test_y[outlier],
             {f"{name} (MSE {model_errors[name][outlier]:.4g})": model_predictions[name][outlier] for name in names},
             "Zn mp-571292 site 000 | held-out test")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
def plot_grid(panels, title, ncol=3):
    nrow = (len(panels) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.8 * ncol, 3.35 * nrow), squeeze=False)
    for ax, panel in zip(axes.flat, panels):
        plot_spectra(ax, *panel)
    for ax in list(axes.flat)[len(panels):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def test_panel(index, label):
    _, material, site = test_keys[index]
    return (ENERGY_GRID, test_y[index], {name: model_predictions[name][index] for name in names},
            f"Zn {material} site {site:03d} | {label}\nExpertXAS MSE {errors[index]:.4g}")

percentiles = (5, 10, 20, 30, 50, 70, 80, 90, 95)
available = list(range(len(errors)))
panels = []
for p in percentiles:
    if not available:
        break
    index = min(available, key=lambda i: abs(errors[i] - np.percentile(errors, p)))
    available.remove(index)
    panels.append(test_panel(index, f"near P{p}"))
plot_grid(panels, "Zn test spectra by ExpertXAS MSE percentile")

p95 = np.percentile(errors, 95)
seen_materials, panels = set(), []
for index in np.argsort(abs(errors - p95), kind="stable"):
    material = test_keys[index][1]
    if material in seen_materials:
        continue
    seen_materials.add(material)
    panels.append(test_panel(index, "near P95"))
    if len(panels) == 9:
        break
plot_grid(panels, "Zn test spectra near P95 | distinct materials")

panels = []
for split in ("test", "val", "train"):
    _, keys, y = base["Zn", split]
    indices = [i for i, key in enumerate(keys) if key in base_iodine]
    if not indices:
        continue
    X, _ = base_features["Zn", split]
    preds = ({name: values[indices] for name, values in model_predictions.items()} if split == "test"
             else {name: predict_head(model, X[indices]) for name, model in model_heads.items()})
    for index, key_index in enumerate(indices):
        _, material, site = keys[key_index]
        target = y[key_index]
        base_error = np.mean((target - train_mean) ** 2)
        curves = {}
        for name, values in preds.items():
            ratio = error_ratio(base_error, np.mean((target - values[index]) ** 2))
            curves[f"{name} ({ratio:.2f}×)"] = values[index]
        panels.append((ENERGY_GRID, target, curves, f"Zn {material} site {site:03d} | {split.upper()}"))
plot_grid(panels, "Iodine-containing canonical Zn spectra | baseline MSE divided by model MSE")

## Evaluate the added iodine cohort

Use the selected augmented UniversalXAS on the experimental validation and test splits. Report eta by absorber where a corresponding training mean exists. A pooled eta would combine different energy grids and baselines.

In [ ]:
training_targets_by_absorber = defaultdict(list)
for element in ELEMENTS:
    training_targets_by_absorber[element].append(base_features[element, "train"][1])
for (absorber, _, _), _, y, _ in extra["train"]:
    training_targets_by_absorber[absorber].append(y[None, :])
training_means = {element: np.concatenate(ys).mean(axis=0) for element, ys in training_targets_by_absorber.items()}

extra_metrics, test_panels = [], []
for split in ("val", "test"):
    rows = extra[split]
    X = np.stack([row[1] for row in rows])
    y = np.stack([row[2] for row in rows])
    predicted = predict_head(model_heads["UniversalXAS"], X)
    error = ((predicted - y) ** 2).mean(axis=1)
    absorber_groups = defaultdict(list)
    for index, (key, _, _, _) in enumerate(rows):
        absorber_groups[key[0]].append(index)
    shown_materials = set()
    for absorber, indices in sorted(absorber_groups.items(), key=lambda item: (-len(item[1]), item[0])):
        baseline = training_means.get(absorber)
        median = float(np.median(error[indices]))
        eta = error_ratio(np.median(((y[indices] - baseline) ** 2).mean(axis=1)), median) if baseline is not None else np.nan
        extra_metrics.append({"split": split, "absorber": absorber, "spectra": len(indices),
                              "materials": len({rows[i][0][1] for i in indices}),
                              "MSE": float(error[indices].mean()), "median MSE": median, "eta": eta})
        if split == "test" and len(test_panels) < 9:
            candidate_indices = [i for i in indices if rows[i][0][1] not in shown_materials]
            if candidate_indices:
                i = min(candidate_indices, key=lambda i: abs(error[i] - median))
                (_, material, site), _, target, grid = rows[i]
                shown_materials.add(material)
                axis = "Energy (eV)" if absorber in ELEMENTS else "Relative energy (eV)"
                title = f"{absorber} {material} site {site:03d} | TEST\nMSE {error[i]:.4g}"
                test_panels.append((grid, target, {"UniversalXAS": predicted[i]}, title, axis))
display(pd.DataFrame(extra_metrics))
plot_grid(test_panels, "Added iodine-bearing test spectra | augmented UniversalXAS")